In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql.functions import col

In [2]:
spark = SparkSession.builder \
    .appName("S3Integration") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.2,com.amazonaws:aws-java-sdk-bundle:1.12.115") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .getOrCreate()

26/03/23 00:32:01 WARN Utils: Your hostname, minh-HP-250-G8-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.1.13 instead (on interface wlo1)
26/03/23 00:32:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/minh/Documents/Docs/Workspace/Projects/us_flight_delay_analytics/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/minh/.ivy2/cache
The jars for the packages stored in: /home/minh/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8fc69efb-e561-4b80-80c9-75e970753369;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.115 in central
:: resolution report :: resolve 356ms :: artifacts dl 16ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.115 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 by [com.amazonaws#aws-java-sdk-bundle;1.12.115] in [default]
	------------------------------------------------------------------

In [3]:
def check_missing_value(df):
    for col_name in df.columns:
        return df.filter(col(col_name).isNull()).count()

In [4]:
df_airport_geo = spark.read \
    .option("header", "true") \
    .csv("s3a://us-flight-delay-analytics-data-lake/raw_flights/airports_geolocation.csv")

26/03/23 00:32:07 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [5]:
df_airport_geo.show()

+---------+--------------------+-------------+-----+-------+--------+----------+
|IATA_CODE|             AIRPORT|         CITY|STATE|COUNTRY|LATITUDE| LONGITUDE|
+---------+--------------------+-------------+-----+-------+--------+----------+
|      ABE|Lehigh Valley Int...|    Allentown|   PA|    USA|40.65236|  -75.4404|
|      ABI|Abilene Regional ...|      Abilene|   TX|    USA|32.41132|  -99.6819|
|      ABQ|Albuquerque Inter...|  Albuquerque|   NM|    USA|35.04022|-106.60919|
|      ABR|Aberdeen Regional...|     Aberdeen|   SD|    USA|45.44906| -98.42183|
|      ABY|Southwest Georgia...|       Albany|   GA|    USA|31.53552| -84.19447|
|      ACK|Nantucket Memoria...|    Nantucket|   MA|    USA|41.25305| -70.06018|
|      ACT|Waco Regional Air...|         Waco|   TX|    USA|31.61129| -97.23052|
|      ACV|      Arcata Airport|Arcata/Eureka|   CA|    USA|40.97812|-124.10862|
|      ACY|Atlantic City Int...|Atlantic City|   NJ|    USA| 39.3642|  -74.4231|
|      ADK|        Adak Airp

In [6]:
check_missing_value(df_airport_geo)

0

In [7]:
df_airport_geo.schema

StructType([StructField('IATA_CODE', StringType(), True), StructField('AIRPORT', StringType(), True), StructField('CITY', StringType(), True), StructField('STATE', StringType(), True), StructField('COUNTRY', StringType(), True), StructField('LATITUDE', StringType(), True), StructField('LONGITUDE', StringType(), True)])

In [8]:
airport_geo_schema = types.StructType([
    types.StructField('IATA_CODE', types.StringType(), True), 
    types.StructField('AIRPORT', types.StringType(), True), 
    types.StructField('CITY', types.StringType(), True), 
    types.StructField('STATE', types.StringType(), True), 
    types.StructField('COUNTRY', types.StringType(), True), 
    types.StructField('LATITUDE', types.DoubleType(), True), 
    types.StructField('LONGITUDE', types.DoubleType(), True)]
)

In [9]:
df_airport_geo_clean = spark.read \
    .option("header", "true") \
    .schema(airport_geo_schema) \
    .csv("s3a://us-flight-delay-analytics-data-lake/raw_flights/airports_geolocation.csv")

In [10]:
df_airport_geo_clean.write.parquet("s3a://us-flight-delay-analytics-data-lake/clean/geo",mode="overwrite")

In [11]:
df_flights = spark.read \
    .option("header", "true") \
    .csv("s3a://us-flight-delay-analytics-data-lake/raw_flights/US_flights_2023.csv")

In [12]:
df_flights_24 = spark.read \
    .option("header", "true") \
    .csv("s3a://us-flight-delay-analytics-data-lake/raw_flights/maj us flight - january 2024.csv")

In [13]:
df_flights.head()

Row(FlightDate='2023-01-02', Day_Of_Week='1', Airline='Endeavor Air', Tail_Number='N605LR', Dep_Airport='BDL', Dep_CityName='Hartford, CT', DepTime_label='Morning', Dep_Delay='-3', Dep_Delay_Tag='0', Dep_Delay_Type='Low <5min', Arr_Airport='LGA', Arr_CityName='New York, NY', Arr_Delay='-12', Arr_Delay_Type='Low <5min', Flight_Duration='56', Distance_type='Short Haul >1500Mi', Delay_Carrier='0', Delay_Weather='0', Delay_NAS='0', Delay_Security='0', Delay_LastAircraft='0', Manufacturer='CANADAIR REGIONAL JET', Model='CRJ', Aicraft_age='16')

In [14]:
df_flights_24.head()

Row(FlightDate='2023-01-02', Day_Of_Week='1', Airline='Endeavor Air', Tail_Number='N605LR', Dep_Airport='BDL', Dep_CityName='Hartford, CT', DepTime_label='Morning', Dep_Delay='-3', Dep_Delay_Tag='0', Dep_Delay_Type='Low <5min', Arr_Airport='LGA', Arr_CityName='New York, NY', Arr_Delay='-12', Arr_Delay_Type='Low <5min', Flight_Duration='56', Distance_type='Short Haul >1500Mi', Delay_Carrier='0', Delay_Weather='0', Delay_NAS='0', Delay_Security='0', Delay_LastAircraft='0', Manufacturer='CANADAIR REGIONAL JET', Model='CRJ', Aicraft_age='16')

In [15]:
check_missing_value(df_flights)

0

In [16]:
check_missing_value(df_flights_24)

0

In [17]:
df_flights.schema

StructType([StructField('FlightDate', StringType(), True), StructField('Day_Of_Week', StringType(), True), StructField('Airline', StringType(), True), StructField('Tail_Number', StringType(), True), StructField('Dep_Airport', StringType(), True), StructField('Dep_CityName', StringType(), True), StructField('DepTime_label', StringType(), True), StructField('Dep_Delay', StringType(), True), StructField('Dep_Delay_Tag', StringType(), True), StructField('Dep_Delay_Type', StringType(), True), StructField('Arr_Airport', StringType(), True), StructField('Arr_CityName', StringType(), True), StructField('Arr_Delay', StringType(), True), StructField('Arr_Delay_Type', StringType(), True), StructField('Flight_Duration', StringType(), True), StructField('Distance_type', StringType(), True), StructField('Delay_Carrier', StringType(), True), StructField('Delay_Weather', StringType(), True), StructField('Delay_NAS', StringType(), True), StructField('Delay_Security', StringType(), True), StructField('D

In [18]:
df_flights_24.schema

StructType([StructField('FlightDate', StringType(), True), StructField('Day_Of_Week', StringType(), True), StructField('Airline', StringType(), True), StructField('Tail_Number', StringType(), True), StructField('Dep_Airport', StringType(), True), StructField('Dep_CityName', StringType(), True), StructField('DepTime_label', StringType(), True), StructField('Dep_Delay', StringType(), True), StructField('Dep_Delay_Tag', StringType(), True), StructField('Dep_Delay_Type', StringType(), True), StructField('Arr_Airport', StringType(), True), StructField('Arr_CityName', StringType(), True), StructField('Arr_Delay', StringType(), True), StructField('Arr_Delay_Type', StringType(), True), StructField('Flight_Duration', StringType(), True), StructField('Distance_type', StringType(), True), StructField('Delay_Carrier', StringType(), True), StructField('Delay_Weather', StringType(), True), StructField('Delay_NAS', StringType(), True), StructField('Delay_Security', StringType(), True), StructField('D

In [19]:
flights_schema = types.StructType([
    types.StructField('FlightDate', types.TimestampType(), True), 
    types.StructField('Day_Of_Week', types.IntegerType(), True), 
    types.StructField('Airline', types.StringType(), True), 
    types.StructField('Tail_Number', types.StringType(), True), 
    types.StructField('Dep_Airport', types.StringType(), True), 
    types.StructField('Dep_CityName', types.StringType(), True), 
    types.StructField('DepTime_label', types.StringType(), True), 
    types.StructField('Dep_Delay', types.IntegerType(), True), 
    types.StructField('Dep_Delay_Tag', types.IntegerType(), True), 
    types.StructField('Dep_Delay_Type', types.StringType(), True), 
    types.StructField('Arr_Airport', types.StringType(), True), 
    types.StructField('Arr_CityName', types.StringType(), True), 
    types.StructField('Arr_Delay', types.IntegerType(), True), 
    types.StructField('Arr_Delay_Type', types.StringType(), True), 
    types.StructField('Flight_Duration', types.IntegerType(), True), 
    types.StructField('Distance_type', types.StringType(), True), 
    types.StructField('Delay_Carrier', types.IntegerType(), True), 
    types.StructField('Delay_Weather', types.IntegerType(), True), 
    types.StructField('Delay_NAS', types.IntegerType(), True), 
    types.StructField('Delay_Security', types.IntegerType(), True), 
    types.StructField('Delay_LastAircraft', types.IntegerType(), True), 
    types.StructField('Manufacturer', types.StringType(), True), 
    types.StructField('Model', types.StringType(), True), 
    types.StructField('Aicraft_age', types.IntegerType(), True)]
)

In [20]:
df_flights_clean = spark.read \
    .option("header", "true") \
    .schema(flights_schema) \
    .csv("s3a://us-flight-delay-analytics-data-lake/raw_flights/US_flights_2023.csv")

In [21]:
df_flights_24_clean = spark.read \
    .option("header", "true") \
    .schema(flights_schema) \
    .csv("s3a://us-flight-delay-analytics-data-lake/raw_flights/maj us flight - january 2024.csv")

In [22]:
df_flights_final_clean = df_flights_clean.unionByName(df_flights_24_clean)

In [23]:
df_flights_final_clean.write.parquet("s3a://us-flight-delay-analytics-data-lake/clean/flights", mode="overwrite")

In [24]:
df_can_div = spark.read \
    .option("header", "true") \
    .csv("s3a://us-flight-delay-analytics-data-lake/raw_flights/Cancelled_Diverted_2023.csv")

In [25]:
df_can_div.head()

Row(FlightDate='2023-01-25', Day_Of_Week='3', Airline='Endeavor Air', Tail_Number='N691CA', Cancelled='1.0', Diverted='0.0', Dep_Airport='JFK', Dep_CityName='New York, NY', DepTime_label='Evening', Dep_Delay='0.0', Dep_Delay_Tag='0', Dep_Delay_Type='No Departure Delay', Arr_Airport='ITH', Arr_CityName='Ithaca/Cortland, NY', Arr_Delay='0.0', Arr_Delay_Type='No Arrival Delay', Flight_Duration='0.0', Distance_type='Short Haul', Delay_Carrier='0.0', Delay_Weather='0.0', Delay_NAS='0.0', Delay_Security='0.0', Delay_LastAircraft='0.0')

In [26]:
check_missing_value(df_can_div)

0

In [27]:
df_can_div.schema

StructType([StructField('FlightDate', StringType(), True), StructField('Day_Of_Week', StringType(), True), StructField('Airline', StringType(), True), StructField('Tail_Number', StringType(), True), StructField('Cancelled', StringType(), True), StructField('Diverted', StringType(), True), StructField('Dep_Airport', StringType(), True), StructField('Dep_CityName', StringType(), True), StructField('DepTime_label', StringType(), True), StructField('Dep_Delay', StringType(), True), StructField('Dep_Delay_Tag', StringType(), True), StructField('Dep_Delay_Type', StringType(), True), StructField('Arr_Airport', StringType(), True), StructField('Arr_CityName', StringType(), True), StructField('Arr_Delay', StringType(), True), StructField('Arr_Delay_Type', StringType(), True), StructField('Flight_Duration', StringType(), True), StructField('Distance_type', StringType(), True), StructField('Delay_Carrier', StringType(), True), StructField('Delay_Weather', StringType(), True), StructField('Delay_N

In [28]:
can_div_schema = types.StructType([
    types.StructField('FlightDate', types.TimestampType(), True), 
    types.StructField('Day_Of_Week', types.IntegerType(), True), 
    types.StructField('Airline', types.StringType(), True), 
    types.StructField('Tail_Number', types.StringType(), True), 
    types.StructField('Cancelled', types.IntegerType(), True), 
    types.StructField('Diverted', types.IntegerType(), True), 
    types.StructField('Dep_Airport', types.StringType(), True), 
    types.StructField('Dep_CityName', types.StringType(), True), 
    types.StructField('DepTime_label', types.StringType(), True), 
    types.StructField('Dep_Delay', types.IntegerType(), True), 
    types.StructField('Dep_Delay_Tag', types.IntegerType(), True), 
    types.StructField('Dep_Delay_Type', types.StringType(), True), 
    types.StructField('Arr_Airport', types.StringType(), True), 
    types.StructField('Arr_CityName', types.StringType(), True), 
    types.StructField('Arr_Delay', types.IntegerType(), True), 
    types.StructField('Arr_Delay_Type', types.StringType(), True), 
    types.StructField('Flight_Duration', types.IntegerType(), True), 
    types.StructField('Distance_type', types.StringType(), True), 
    types.StructField('Delay_Carrier', types.IntegerType(), True), 
    types.StructField('Delay_Weather', types.IntegerType(), True), 
    types.StructField('Delay_NAS', types.IntegerType(), True), 
    types.StructField('Delay_Security', types.IntegerType(), True), 
    types.StructField('Delay_LastAircraft', types.IntegerType(), True)]
)

In [29]:
df_can_div_clean = spark.read \
    .option("header", "true") \
    .schema(can_div_schema) \
    .csv("s3a://us-flight-delay-analytics-data-lake/raw_flights/Cancelled_Diverted_2023.csv")

In [30]:
df_can_div_clean.write.parquet("s3a://us-flight-delay-analytics-data-lake/clean/cancelled_diverted", mode="overwrite")

In [31]:
df_weather = spark.read \
    .option("header", "true") \
    .csv("s3a://us-flight-delay-analytics-data-lake/raw_flights/weather_meteo_by_airport.csv")

In [32]:
df_weather.show()

+----------+----+----+----+----+----+-----+----+------+----------+
|      time|tavg|tmin|tmax|prcp|snow| wdir|wspd|  pres|airport_id|
+----------+----+----+----+----+----+-----+----+------+----------+
|2023-01-01| 8.1| 2.2|11.7| 0.0| 0.0|278.0| 9.7|1013.8|       ABE|
|2023-01-02| 5.4| 0.0|11.7| 0.0| 0.0|353.0| 3.6|1019.6|       ABE|
|2023-01-03| 8.4| 7.2| 9.4|15.2| 0.0| 50.0| 5.0|1013.9|       ABE|
|2023-01-04|11.1| 6.7|17.2| 0.0| 0.0|302.0| 4.7|1009.8|       ABE|
|2023-01-05|12.7| 6.7|14.4| 7.9| 0.0|292.0| 7.2|1013.0|       ABE|
|2023-01-06| 5.8| 2.8| 7.2| 5.8| 0.0|308.0| 9.0|1016.6|       ABE|
|2023-01-07| 3.8| 1.7| 6.7| 0.0| 0.0|285.0|11.2|1022.2|       ABE|
|2023-01-08| 2.2|-2.7| 3.9| 0.0| 0.0|346.0| 6.8|1024.5|       ABE|
|2023-01-09| 2.3|-1.0| 5.0| 0.0| 0.0|271.0|11.9|1015.2|       ABE|
|2023-01-10| 1.9|-1.6| 4.4| 0.0| 0.0|250.0| 9.4|1017.4|       ABE|
|2023-01-11| 2.3|-3.2| 5.0| 0.0| 0.0| 91.0| 9.0|1022.6|       ABE|
|2023-01-12| 4.1| 1.1|11.1| 4.1| 0.0| 94.0|11.2|1015.6|       

In [33]:
check_missing_value(df_weather)

0

In [34]:
df_weather.schema

StructType([StructField('time', StringType(), True), StructField('tavg', StringType(), True), StructField('tmin', StringType(), True), StructField('tmax', StringType(), True), StructField('prcp', StringType(), True), StructField('snow', StringType(), True), StructField('wdir', StringType(), True), StructField('wspd', StringType(), True), StructField('pres', StringType(), True), StructField('airport_id', StringType(), True)])

In [35]:
weather_schema = types.StructType([
    types.StructField('time', types.TimestampType(), True), 
    types.StructField('tavg', types.FloatType(), True), 
    types.StructField('tmin', types.FloatType(), True), 
    types.StructField('tmax', types.FloatType(), True), 
    types.StructField('prcp', types.FloatType(), True), 
    types.StructField('snow', types.FloatType(), True), 
    types.StructField('wdir', types.FloatType(), True), 
    types.StructField('wspd', types.FloatType(), True), 
    types.StructField('pres', types.FloatType(), True), 
    types.StructField('airport_id', types.StringType(), True)]
)

In [36]:
df_weather_clean = spark.read \
    .option("header", "true") \
    .schema(weather_schema) \
    .csv("s3a://us-flight-delay-analytics-data-lake/raw_flights/weather_meteo_by_airport.csv")

In [37]:
df_weather_clean.write.parquet("s3a://us-flight-delay-analytics-data-lake/clean/weather", mode="overwrite")

In [ ]:
spark.stop()